# 19.22 Red teaming & evaluation

Red teaming turns vague safety worries into adversarial test cases, metrics, and release gates.

This notebook turns failures into denominated scenario risks, weighted scores, and release gates. The D5 pitfall counts found bugs without denominators, then fixes the scorecard with predeclared gates and regression tests.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 1919
rng = np.random.default_rng(SEED)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def _standardize(X):
    scaler = StandardScaler()
    return scaler.fit_transform(np.asarray(X, dtype=float))


def _binary_target(y):
    y = np.asarray(y)
    classes = np.unique(y)
    if len(classes) == 2:
        return (y == classes[-1]).astype(int)
    return (y == classes[-1]).astype(int)


def clf_ladder():
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def logistic_baseline(x_tr, y_tr, x_te):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


def fit_logistic(X, y):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(X, y)
    return clf


## Build a red-team score once on D1\n\nThe lesson formula is $$S=\sum_j w_j r_j.$$\nThe lesson risks satisfy $0.050+0.100+0.200=0.350$, and the guard with knob $0.600$ gives $0.350+0.600\cdot0.350=0.560$.

In [ ]:

def red_team_score(risks, weights, gates):
    risks = np.asarray(risks, dtype=float)
    weights = np.asarray(weights, dtype=float)
    score = float(np.sum(weights * risks))
    passed = bool(np.all(risks <= np.asarray(gates, dtype=float)))
    return score, passed


def risks_from_counts(failures, denominators):
    failures = np.asarray(failures, dtype=float)
    denominators = np.maximum(np.asarray(denominators, dtype=float), 1.0)
    return failures / denominators

risks = np.array([0.050, 0.100, 0.200])
weights = np.array([1.0, 1.0, 1.0])
gates = np.array([0.10, 0.12, 0.25])
score, passed = red_team_score(risks, weights, gates)
guarded = score + 0.600 * np.abs(risks).sum()
assert round(score, 3) == 0.350
assert round(guarded, 3) == 0.560
print("risk score", round(score, 3))
print("guarded score", round(guarded, 3))
print("passes gates", passed)


Create a reusable scorecard: clean error, stress error, subgroup gap, privacy-like confidence risk, and regression failures.

In [ ]:

def make_stress_view(X, stress, seed):
    local_rng = np.random.default_rng(seed)
    Xs = _standardize(X)
    noise = local_rng.normal(0.0, stress, Xs.shape)
    return Xs + noise


def red_team_card(X, y, stress, seed):
    yb = _binary_target(y)
    Xs = _standardize(X)
    x_tr, x_te, y_tr, y_te = train_test_split(Xs, yb, test_size=0.4, random_state=seed, stratify=yb)
    model = fit_logistic(x_tr, y_tr)
    clean_pred = model.predict(x_te)
    clean_error = 1.0 - accuracy_score(y_te, clean_pred)
    x_stress = x_te + np.random.default_rng(seed).normal(0.0, stress, x_te.shape)
    stress_pred = model.predict(x_stress)
    stress_error = 1.0 - accuracy_score(y_te, stress_pred)
    group = x_te[:, 0] > np.median(x_te[:, 0])
    group_a = 1.0 - accuracy_score(y_te[group], stress_pred[group])
    group_b = 1.0 - accuracy_score(y_te[~group], stress_pred[~group])
    subgroup_gap = abs(group_a - group_b)
    confidence = model.predict_proba(x_te).max(axis=1)
    privacy_like_risk = float(np.mean(confidence > 0.95))
    regression_failures = float(np.mean(clean_pred != stress_pred))
    return np.array([clean_error, stress_error, subgroup_gap, privacy_like_risk, regression_failures])

card = red_team_card(clf_ladder()[0][1], clf_ladder()[0][2], 0.05, SEED)
print("D1 card", np.round(card, 3))


## Dataset ladder

The F15 ladder is audited with a five-scenario red-team pack and rising stress.

In [ ]:

redteam_specs = [
    ("D1 linear toy", 0.00),
    ("D2 noisy perturbations", 0.10),
    ("D3 nonlinear stress", 0.20),
    ("D4 image-like stress", 0.35),
    ("D5 shifted attacked biased", 0.55),
]
redteam_rungs = []
for idx, ((base_name, X, y), (name, stress)) in enumerate(zip(clf_ladder(), redteam_specs)):
    card = red_team_card(X, y, stress, SEED + idx)
    redteam_rungs.append((name, card, stress))
    print(name, "stress", stress, "risks", np.round(card, 3))


## Run the weighted red-team score across D1-D5

In [ ]:

redteam_weights = np.array([0.8, 1.2, 1.4, 0.7, 1.0])
redteam_gates = np.array([0.25, 0.35, 0.25, 0.60, 0.30])
redteam_results = []
for name, card, stress in redteam_rungs:
    score, passed = red_team_score(card, redteam_weights, redteam_gates)
    robust_accuracy = 1.0 - card[1]
    redteam_results.append((name, score, robust_accuracy, passed, card))

print("rung | risk_score | robust_acc | passed")
for row in redteam_results:
    print(row[0], round(row[1], 3), round(row[2], 3), row[3])


## Results visualization

Left: per-rung risk cards. Right: weighted risk score as stress rises.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()
labels = ["clean", "stress", "gap", "privacy", "regress"]
for ax, row in zip(axes[:5], redteam_results):
    name, score, robust_accuracy, passed, card = row
    ax.bar(labels, card, color="slateblue")
    ax.axhline(0.25, color="tomato", linestyle="--", linewidth=1)
    ax.set_title(name)
    ax.tick_params(axis="x", rotation=35)
axes[5].plot(range(1, 6), [row[1] for row in redteam_results], marker="o")
axes[5].set_xticks(range(1, 6))
axes[5].set_xlabel("rung")
axes[5].set_ylabel("weighted risk score")
axes[5].set_title("Risk vs stress")
plt.tight_layout()


## Pitfall on D5: counting only found bugs

The wrong score uses raw found-bug counts, so larger test packs look worse even if rates improve. The fix divides by scenario denominators, applies gates, and keeps regression failures in the suite.

In [ ]:

found_failures = np.array([2, 5, 4, 3, 6])
scenario_denominators = np.array([100, 100, 40, 80, 120])
wrong_count_score = float(found_failures.sum())
fixed_risks = risks_from_counts(found_failures, scenario_denominators)
fixed_score, fixed_passed = red_team_score(fixed_risks, redteam_weights, redteam_gates)
print("wrong raw bug count", round(wrong_count_score, 3))
print("fixed denominated risks", np.round(fixed_risks, 3))
print("fixed weighted score", round(fixed_score, 3))
print("passes predeclared gates", fixed_passed)


## Evaluate it + Practice

- Metric: weighted risk score and robust accuracy
- No-skill baseline: raw found-bug count without denominators
- Cheap sanity check: zero failures should give zero score and pass gates
- Ablation: remove denominators or regression scenarios and the score should become misleading
- Failure signals: a gate failure, rising stress risk, or returning fixed failure

Practice prompts:
1. Change the D5 stress level and predict which metric should move first.

2. Add one subgroup or environment slice and explain whether the conclusion changes.

3. Replace the default logistic model with another CPU-only sklearn classifier and compare the same metric.